In [0]:
INSERT OVERWRITE `nyc-mobility`.nyc_silver.taxi_zones_clean
SELECT
  CAST(LocationID AS INT) AS location_id,
  CASE 
    WHEN TRIM(Borough) IN ('N/A', 'Unknown', '') THEN 'Unknown'
    ELSE TRIM(Borough)
  END AS borough,
  TRIM(Zone) AS zone,
  CASE 
    WHEN TRIM(service_zone) IN ('N/A', 'Unknown', '') THEN 'Unknown'
    ELSE TRIM(service_zone)
  END AS service_zone,
  ingestion_time
FROM `nyc-mobility`.nyc_bronze.taxi_zones;

-- PROFILE: ROW COUNT

SELECT COUNT(*) AS total_rows
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean;

-- PROFILE: NULL COUNTS PER COLUMN

SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(location_id) AS null_location_id,
  COUNT(*) - COUNT(borough) AS null_borough,
  COUNT(*) - COUNT(zone) AS null_zone,
  COUNT(*) - COUNT(service_zone) AS null_service_zone
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean;

-- PROFILE: UNIQUENESS / CARDINALITY OF location_id (primary key check)

SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT location_id) AS distinct_location_ids
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean;

-- PROFILE: DUPLICATE location_id ROWS (should return zero rows)

SELECT location_id, COUNT(*) AS occurrences
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
GROUP BY location_id
HAVING COUNT(*) > 1;

-- PROFILE: VALUE DISTRIBUTION — borough

SELECT borough, COUNT(*) AS row_count
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
GROUP BY borough
ORDER BY row_count DESC;

-- PROFILE: VALUE DISTRIBUTION — service_zone

SELECT service_zone, COUNT(*) AS row_count
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
GROUP BY service_zone
ORDER BY row_count DESC;

-- PROFILE: DOES 'Unknown' IN service_zone CORRELATE WITH 'Unknown' IN borough?

SELECT borough, service_zone, COUNT(*) AS row_count
FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
WHERE service_zone IN ('N/A', 'Unknown', '')
   OR borough IN ('N/A', 'Unknown', '')
GROUP BY borough, service_zone
ORDER BY row_count DESC;

-- QUALITY GATE: FAIL-LOUD ASSERTION ON location_id UNIQUENESS
-- Returns a row (alert) only if duplicates are detected

SELECT 'DUPLICATE LOCATION_ID DETECTED IN SILVER' AS alert
WHERE (
  SELECT COUNT(*) FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
) != (
  SELECT COUNT(DISTINCT location_id) FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
);

-- QUALITY GATE: FAIL-LOUD ASSERTION ON NULL location_id
-- Returns a row (alert) only if any location_id is null

SELECT 'NULL LOCATION_ID DETECTED IN SILVER' AS alert
WHERE EXISTS (
  SELECT 1
  FROM `nyc-mobility`.nyc_silver.taxi_zones_clean
  WHERE location_id IS NULL
);